In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from itertools import zip_longest
import matplotlib.pyplot as plt
import shutil

Define the tool function

In [ ]:
def count_atoms(df, target_atom):
    """
    Count the number of specific atoms in a DataFrame
    
    Parameters:
        df (pd.DataFrame): DataFrame containing atomic data
        target_atom (str): Target atomic symbol to count (e.g., 'C', 'O', 'Zr')
    
    Returns:
        int: Count of target atoms
    """
    # Ensure atomic symbols are strings and strip whitespace
    df['atoms'] = df['atoms'].astype(str).str.strip()
    
    # Count atoms matching target
    return df[df['atoms'] == target_atom].shape[0]

In [ ]:
def extract_simulation_conditions(file_path):
    """
    Extract simulation conditions from parent folder name in the given file path.
    
    Args:
        file_path: Full file path containing parameters in parent folder name 
                 (e.g., ".../Zr_1.0_FA_18500.0_.../file.mol2")
    
    Returns:
        Dictionary containing parsed parameters (e.g., {'Zr6': 1.0, 'FA': 18500.0, ...})
        Returns empty dict if parsing fails
    """
    try:
        # 1. Extract immediate parent folder name from path
        folder_name = os.path.basename(os.path.dirname(file_path))
        
        # 2. Remove timestamp suffix (format: _YYYY-MM-DD HH:MM:SS)
        clean_name = re.sub(r"_\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$", "", folder_name)
        
        # 3. List of target parameters to extract (extendable)
        target_params = ["Zr", "FA", "L", "Ratio", "Step", "Index", "SC", "KC"]
        # Build regex pattern to match any of the target parameters followed by value
        pattern = r"(" + "|".join(target_params) + r")_([^_]+)"
        
        # 4. Extract key-value pairs and convert numeric values
        params = {}
        for key, value in re.findall(pattern, clean_name):
            try:
                # Handle scientific notation (e.g., 1e8.0) and regular floats
                # Replace lowercase 'e' with uppercase 'E' for consistency
                params[key] = float(value.replace('e', 'E'))  
            except ValueError:
                # Keep original string if not convertible to float
                params[key] = value  
        
        return params

    except Exception as e:
        print(f"Parameter extraction failed: {str(e)}")
        return {}

In [ ]:
def extract_block_number(filename: str) -> int:
    """Extract the block number from the filename for sorting purposes."""
    match = re.search(r"block_(\d+)\.mol2$", filename)
    return int(match.group(1)) if match else 0

In [ ]:
# def compare_geometry(coords1, coords2):
#     """
#     Compare two sets of 3D coordinates and compute RMSD.
#     Only compares geometry, ignores element types.

#     Args:
#         coords1 (np.ndarray): shape (N,3)
#         coords2 (np.ndarray): shape (N,3)

#     Returns:
#         float: RMSD between the two coordinate sets, or np.inf if failed
#     """
#     try:
#         coords1 = np.asarray(coords1, dtype=float)
#         coords2 = np.asarray(coords2, dtype=float)

#         if coords1.shape != coords2.shape:
#             raise ValueError(f"Shape mismatch: {coords1.shape} vs {coords2.shape}")

#         # 平移到质心
#         c1_centroid = coords1.mean(axis=0)
#         c2_centroid = coords2.mean(axis=0)
#         c1_centered = coords1 - c1_centroid
#         c2_centered = coords2 - c2_centroid

#         # 使用 Kabsch 算法求最优旋转
#         H = c1_centered.T @ c2_centered
#         U, S, Vt = np.linalg.svd(H)
#         R = Vt.T @ U.T

#         # 修正反射问题
#         if np.linalg.det(R) < 0:
#             Vt[-1,:] *= -1
#             R = Vt.T @ U.T

#         c1_rot = c1_centered @ R

#         # 计算 RMSD
#         diff = c1_rot - c2_centered
#         rmsd = np.sqrt((diff**2).sum() / coords1.shape[0])
#         return rmsd

#     except Exception as e:
#         print(f"[RMSD error]: {e}")
#         return float('inf')

In [ ]:
def radial_histograms(coords, elems, reference_mask=None, bins=np.linspace(0,50,51)):
    """
    Compute radial histograms for each element using a reference atom centroid as origin.
    
    Parameters:
    - coords: N×3 numpy array of atomic coordinates
    - elems: array of element symbols (dtype=object/str)
    - reference_mask: boolean mask specifying reference atoms (e.g., Zr); if None or empty, use overall centroid
    - bins: array specifying bin edges for distance histogram
    
    Returns:
    - dict {elem: normalized_hist (numpy array)}
    """
    if reference_mask is None or reference_mask.sum() == 0:
        ref = coords.mean(axis=0)
    else:
        ref = coords[reference_mask].mean(axis=0)
    dists = np.linalg.norm(coords - ref, axis=1)

    hist_dict = {}
    for e in np.unique(elems):
        h, _ = np.histogram(dists[elems == e], bins=bins)
        h = h.astype(float)
        if h.sum() > 0:
            h /= h.sum()  # normalize histogram
        hist_dict[e] = h
    return hist_dict

def hist_similarity(h1, h2):
    """
    Compare two radial histogram dictionaries using cosine similarity (0..1).
    
    Returns the average similarity across all elements.
    """
    sims = []
    for e in set(h1.keys()) | set(h2.keys()):
        a = h1.get(e, np.zeros_like(next(iter(h1.values()))))
        b = h2.get(e, np.zeros_like(next(iter(h2.values()))))
        na = np.linalg.norm(a) + 1e-12
        nb = np.linalg.norm(b) + 1e-12
        sims.append(np.dot(a, b) / (na * nb))
    return np.mean(sims) if sims else 0.0

Main loop

In [ ]:
data_dir = os.getcwd()
all_records = []
unique_structure_pool={}
# Group the mol2 files
all_mol2_files = []
for name in os.listdir(data_dir):
    dirpath = os.path.join(data_dir, name)
    if os.path.isdir(dirpath): 
        for file in os.listdir(dirpath):
            # if file.endswith(".mol2")&("block" in file):
            if file.endswith(".mol2")&("entity_number" in file):
                mol2_file=os.path.join(dirpath, file)

                # Step1:extract the experimental condition
                conditions = extract_simulation_conditions(mol2_file)
                existing_record = next((r for r in all_records if all(r[k]==v for k,v in conditions.items())), None)
                if existing_record is None:
                    record = conditions.copy()
                    # initial the entity_{i} columns
                    for i in range(1, 30):
                        record[f"entity_{i}"] = None
                    all_records.append(record)
                    existing_record = record

                # Load the data
                data = pd.read_csv(mol2_file,
                                   header=None,
                                   skiprows=8, 
                                   delim_whitespace=True,
                                   usecols=[i for i in range(1,5)])
                data.columns = ['atoms', 'x', 'y','z']
                data=data.iloc[:-1]

                # Count the number of SBUs and BTBs
                Zr_count=count_atoms(data,"Zr")
                C_count=count_atoms(data,"C")
                SBU_count = int(Zr_count / 6)
                BTB_count = int((C_count - 12 * SBU_count) / 27)
                entity_count=SBU_count+BTB_count
                if entity_count not in unique_structure_pool:
                    unique_structure_pool[entity_count] = []

                # Extract the coordiantes
                data_cleaned = data.drop_duplicates()
                elems = data_cleaned['atoms'].values
                coords = data_cleaned[['x','y','z']].values.astype(float)

                assigned_uid = None
                for uid, rep in enumerate(unique_structure_pool[entity_count],start=1):
                    # Firstly compare the numbet of BTBs and Zr6s. If the number are not the same, it will be recognized as the unique structure.
                    if rep['SBU_count'] != SBU_count or rep['BTB_count'] != BTB_count:
                        continue

                    # Secondly compare the geometry. 
                    # rmsd = compare_geometry(coords, rep['coords'])
                    # if rmsd < RMSD_THRESH: 
                    #     assigned_uid = uid 
                    #     break
                    h1 = radial_histograms(coords, elems, reference_mask=(elems == 'Zr'))
                    h2 = radial_histograms(rep['coords'], rep['elems'], reference_mask=(rep['elems'] == 'Zr'))
                    sim = hist_similarity(h1, h2)

                    if sim > 0.8:  # 阈值可调，比如 0.75~0.9
                        assigned_uid = uid
                        break

                if assigned_uid is None:
                    unique_structure_pool[entity_count].append({
                        'SBU_count':  SBU_count,
                        'BTB_count': BTB_count,
                        'coords': coords,
                        'elems': elems,
                        'file': mol2_file
                    })
                    new_uid = len(unique_structure_pool[entity_count])
                    assigned_uid=new_uid
                    out_dir = os.path.join(data_dir, "unique_structures", f"entity_{entity_count}")
                    os.makedirs(out_dir, exist_ok=True)
                    new_file = os.path.join(out_dir, f"unique_structure_{assigned_uid}.mol2")
                    shutil.copy(mol2_file, new_file)
                existing_record[f"entity_{entity_count}"] = assigned_uid

In [ ]:
df = pd.DataFrame(all_records)
df.head(100)

In [ ]:
print(all_records)

In [ ]:
for i in range(1, 30):
    col_name = f'entity_{i}'
    count = df[col_name].nunique(dropna=True)
    print(f"{col_name}: {count} unique structures")

In [ ]:
sort_cols = ["Zr","FA","L","Ratio","Step","SC","KC","Index"]
df_sorted = df.sort_values(by=sort_cols)

key_cols = ["Zr","FA","L","Ratio","Step","SC","KC","Index"]
entity_cols = [c for c in df_sorted.columns if c not in key_cols]
df_sorted = df_sorted[key_cols + entity_cols]
df_sorted = df_sorted.reset_index(drop=True)

df_sorted.head(100)

In [ ]:
# Save the final result
output_csv = "unique_structures_result.csv"
df_sorted.to_csv(output_csv, index=False)
print(f"Saved to {output_csv}")

Visualize the result

In [ ]:
import os
import re
import pickle
import numpy as np
import pandas as pd
from collections import defaultdict
import numpy as np
from scipy.optimize import curve_fit
from scipy.spatial import ConvexHull
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

In [ ]:
def equal_aspect(ax, xyz):
    mins, maxs = xyz.min(0), xyz.max(0)
    span = (maxs - mins).max()
    centre = (maxs + mins) / 2
    ax.set_xlim(centre[0]-span/2, centre[0]+span/2)
    ax.set_ylim(centre[1]-span/2, centre[1]+span/2)
    ax.set_zlim(centre[2]-span/2, centre[2]+span/2)
    try:
        ax.set_box_aspect([1,1,1])
    except AttributeError:
        pass
# ──────────────────────────────────────────────────────────────────────
def draw_polyhedron(ax, verts, face_col="royalblue", alpha=0.80,
                    edge_col="k", lw=0.5, scale=1.5):          # <— NEW arg
    """
    Draw the convex hull of `verts` (N×3 array) as a translucent solid,
    optionally isotropically scaled about its centroid.

    Parameters
    ----------
    ax        : matplotlib 3-D axis
    verts     : (N,3) array – original vertex coordinates
    face_col  : face colour
    alpha     : face transparency
    edge_col  : edge colour
    lw        : edge line-width
    scale     : isotropic scale factor relative to the centroid (default 1.5)
    """
    # ---- enlarge hull about its own centre ---------------------------
    c      = verts.mean(axis=0)
    verts_ = c + (verts - c) * scale        # 50 % larger

    # ---- build a convex-hull surface ---------------------------------
    hull   = ConvexHull(verts_)
    faces  = [verts_[simp] for simp in hull.simplices]

    poly   = Poly3DCollection(
                faces, facecolor=face_col, edgecolor=edge_col,
                linewidths=lw, alpha=alpha, zorder=2)
    ax.add_collection3d(poly)
    
def zr6_clusters(elems, xyz):
    """
    Return list[np.ndarray] where every item is a (6, 3) array containing the
    Cartesian coordinates of one consecutive run of six Zr atoms.
    """
    clust, i, n = [], 0, len(elems)
    while i < n:
        if elems[i] == "Zr" and np.all(elems[i:i+6] == "Zr"):
            clust.append(xyz[i:i+6])
            i += 6
        else:
            i += 1
    return clust


# ─────────────────────────────────────────────────────────────────────────────
# Extra helpers for Zr6 + ligand analysis
# ─────────────────────────────────────────────────────────────────────────────
BOND_CUT    = 1.8   # Å – covalent-bond distance threshold
C_THRESHOLD = 5     # ≥ this many C atoms ⇒ “large” ligand component
palette = [
    "orange", "limegreen", "cyan", "magenta", "gold",
    "deepskyblue", "orchid", "salmon", "turquoise", "yellow"
]

def zr6_centroids(elems, xyz):
    """Centroid of every *run* of exactly six consecutive Zr atoms."""
    cents, i, n = [], 0, len(elems)
    while i < n:
        if elems[i] == "Zr" and np.all(elems[i:i+6] == "Zr"):
            cents.append(xyz[i:i+6].mean(0))
            i += 6
        else:
            i += 1
    return np.asarray(cents, np.float32)

def _components(coords, cutoff=BOND_CUT):
    """Connected components by simple distance clustering."""
    N, seen, comps = len(coords), np.zeros(len(coords), bool), []
    for i in range(N):
        if seen[i]:
            continue
        stack, comp = [i], []
        seen[i] = True
        while stack:
            v = stack.pop()
            comp.append(v)
            d2 = np.sum((coords - coords[v])**2, 1)
            nbrs = np.where((d2 < cutoff**2) & (~seen))[0]
            seen[nbrs] = True
            stack.extend(nbrs.tolist())
        comps.append(comp)
    return comps

def component_O_indices(xyz, elems):
    """
    Return {comp_id : list_of_global_O_atom_indices} for ligand components
    that have > C_THRESHOLD carbons.
    """
    non_zr_idx  = np.where(elems != "Zr")[0]
    coords      = xyz[non_zr_idx]
    sub_elems   = elems[non_zr_idx]

    comp_O = {}
    for cid, comp in enumerate(_components(coords)):
        nC = np.count_nonzero(sub_elems[comp] == "C")
        if nC > C_THRESHOLD:
            o_idx = [non_zr_idx[i] for i in comp if sub_elems[i] == "O"]
            if o_idx:
                comp_O[cid] = o_idx
    return comp_O

# ──────────────────────────────────────────────────────────────────
# Helper: choose an s (points²) that stays in [s_min, s_max] but
#         shrinks roughly like 1/n  when n grows.
# ──────────────────────────────────────────────────────────────────
def autosize(n, *, s_min=6, s_max=80, ref_n=5):
    """
    n       : number of points you will scatter
    s_min   : never go below this marker area
    s_max   : never go above this marker area
    ref_n   : ‘reference’ count that maps to ~s_max
    """
    s = s_max * ref_n / max(n, 1)
    return np.clip(s, s_min, s_max)

In [ ]:
data_dir=os.path.join(os.getcwd(),"unique_structures")
for sub_dir in os.listdir(data_dir):
    dirpath = os.path.join(data_dir, sub_dir)
    if os.path.isdir(dirpath):
        mol2_files = [os.path.join(dirpath,f) for f in os.listdir(dirpath) if (f.endswith(".mol2"))]
        
        for mol2_file in mol2_files:
            base_name = os.path.splitext(os.path.basename(mol2_file))[0]
            # Load the data
            data = pd.read_csv(mol2_file,
                               header=None,
                               skiprows=8, 
                               delim_whitespace=True,
                               usecols=[i for i in range(1,5)])
            data.columns = ['atoms', 'x', 'y','z']
            data=data.iloc[:-1]
            data= data[data['atoms'] != 'H'].copy()
            elems = data['atoms'].values
            xyz = data[['x','y','z']].values.astype(float)
    

            zr_cent = zr6_centroids(elems, xyz)           # (M,3) or empty
            comp_O  = component_O_indices(xyz, elems)     # {cid: [indices]}

            # collect all O indices we will highlight so we *don’t* double-plot them
            highlight_O = {idx for lst in comp_O.values() for idx in lst}

            # ------------- plot twice from two viewpoints -------------------------
            fig = plt.figure(figsize=(4, 4))
            ax  = fig.add_subplot(111, projection="3d")
            print(0.1*autosize(len(zr_cent), s_min=0.01, s_max=150))

            # (2-3) each ligand component
            for cid, o_idx in comp_O.items():
                col  = palette[cid % len(palette)]
                pts  = xyz[o_idx]
                n_O  = len(o_idx)
                size = 0.1*autosize(len(zr_cent), s_min=0.1, s_max=150)                                # ★ NEW

                # # O atoms
                # ax.scatter(
                #     pts[:, 0], pts[:, 1], pts[:, 2],
                #     color="red", alpha=0.5, depthshade=False,
                #     s=size*0.1                                           # ★ NEW
                # )

                # ligand centroid (keep comfortably visible)
                cen = pts.mean(0)
                ax.scatter(
                    *cen, marker="x", color="black",
                    s=size * 0.1                          # ★ NEW
                )

                # spokes
                for p in pts:
                    ax.plot([p[0], cen[0]], [p[1], cen[1]], [p[2], cen[2]],
                            color="red", alpha=0.3)

            zr_clusters = zr6_clusters(elems, xyz)
            for cl in zr_clusters:
                draw_polyhedron(ax, cl)  

            # tidy up view
            equal_aspect(ax, xyz)

            # ---- save several viewpoints ----------------------------------------------
            views = [
                ("A",  ax.elev,        ax.azim        ),          # original view
                ("B",  ax.elev,        ax.azim + 90   ),          # +90° azimuth
                ("C",  ax.elev,        ax.azim + 180  ),          # +180° azimuth
                ("D",  ax.elev,        ax.azim + 270  ),          # +270° azimuth
                ("E",  ax.elev + 90,   ax.azim        ),          # top-down
                ("F",  ax.elev - 30,   ax.azim + 45   ),          # oblique example
            ]

            for suf, elev, azim in views:
                ax.view_init(elev=elev, azim=azim)
                out_fname = os.path.join(dirpath, f"{base_name}_{suf}.png")
                plt.savefig(out_fname, dpi=300, bbox_inches="tight")
            
            plt.close(fig)  

Plot the paper picture

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from matplotlib import rcParams
import os
rcParams.update({
    "font.family": "serif",
    "font.size": 28,
    "axes.linewidth": 1.2,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "legend.frameon": False,
    "axes.labelweight": "bold"
})

In [ ]:
save_dir=os.path.join(os.getcwd(),"unique_structures")
data = pd.read_csv("unique_structures_result.csv")
data.fillna(0, inplace=True)

entity_cols = [c for c in data.columns if c.startswith("entity_")]

# ==========================
# Step 1: Calculate probability per entity column
# ==========================
prob_dict = {}  # key: entity_i, value: {uid: probability}

for col in entity_cols:
    uids = data[col].astype(int).tolist()  # convert to int, 0 included
    counter = Counter(uids)
    total = sum(counter.values())
    prob_dict[col] = {uid: count/total for uid, count in counter.items()}

all_uids = sorted({uid for probs in prob_dict.values() for uid in probs})
x_labels = [int(c.split('_')[1]) for c in entity_cols]
bottom = [0]*len(entity_cols)
colors = plt.cm.tab20.colors

fig, ax = plt.subplots(figsize=(30,6))

for i, uid in enumerate(all_uids):
    heights = [prob_dict[col].get(uid, 0) for col in entity_cols]
    ax.bar(x_labels, heights, bottom=bottom,
           color=colors[i % len(colors)], label=f'UID {uid}')
    bottom = [b+h for b,h in zip(bottom, heights)]

ax.set_xlabel("Entity Position")
ax.set_ylabel("Probability")
ax.set_xticks(x_labels)
ax.set_xticklabels(x_labels)  # show only numeric positions
ax.legend(ncol=2, bbox_to_anchor=(1.05,1), loc='upper left')
filename = os.path.join(save_dir, f"summary.png")
plt.savefig(filename, dpi=300)
plt.close(fig)

In [ ]:
save_dir=os.path.join(os.getcwd(),"unique_structures")
for col in entity_cols:
    probs = prob_dict[col]
    uids = sorted(probs.keys())
    heights = [probs[uid] for uid in uids]

    fig, ax = plt.subplots(figsize=(18,6))
    bars = ax.bar(uids, heights, color=[colors[i % len(colors)] for i in range(len(uids))])

    # Find the highest probability bar
    max_idx = heights.index(max(heights))
    max_uid = uids[max_idx]
    max_height = heights[max_idx]

    # Add label above the highest bar
    ax.text(max_uid, max_height + 0.01, f"{max_height:.2f}", ha='center', va='bottom', fontsize=20, fontweight='bold')

    ax.set_xlabel("UID",weight="bold")
    ax.set_ylabel("Probability",weight="bold")
    ax.set_xticks(uids)
    ax.set_xticklabels(uids)
    plt.tight_layout()
    
    # Save figure using the column name as file name
    filename = os.path.join(save_dir, f"{col}.png")
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close(fig)

In [ ]:
data_dir = "_combined_hkust1_e1_to_50_pairtol_0p2"
save_dir = data_dir
data = pd.read_csv(os.path.join(data_dir, "group_summary.csv"))

# Keep only fields needed for plotting
plot_data = data[["entity_number", "group_id", "group_fraction"]].copy()
plot_data["entity_number"] = plot_data["entity_number"].astype(int)
plot_data["group_fraction"] = plot_data["group_fraction"].astype(float)
plot_data = plot_data[plot_data["entity_number"] <= 30]

x_labels = list(range(1, 31))

def _group_sort_key(gid):
    s = str(gid)
    digits = ''.join(ch for ch in s if ch.isdigit())
    return (0, int(digits)) if digits else (1, s)

group_ids = sorted(plot_data["group_id"].unique(), key=_group_sort_key)
bottom = [0.0] * len(x_labels)
colors = plt.cm.tab20.colors

fig, ax = plt.subplots(figsize=(30, 10))

for i, gid in enumerate(group_ids):
    sub = plot_data[plot_data["group_id"] == gid].set_index("entity_number")["group_fraction"].to_dict()
    heights = [sub.get(x, 0.0) for x in x_labels]
    ax.bar(x_labels, heights, bottom=bottom,
           color=colors[i % len(colors)], label=f"{gid}")
    bottom = [b + h for b, h in zip(bottom, heights)]

ax.set_xlabel("Entity Number")
ax.set_ylabel("Probability")
ax.set_xticks(x_labels)
ax.set_xticklabels(x_labels)
ax.set_xlim(0.5, 20.5)
ax.legend(ncol=2, bbox_to_anchor=(1.05, 1), loc="upper left", title="Group ID")
plt.tight_layout()

filename = os.path.join(save_dir, "group_summary_prob_entity_1_to_30_combined_hkust1_e1_to_50_pairtol_0p2.png")
plt.savefig(filename, dpi=300)
plt.close(fig)